# 🎬 ROTBOTS — Effects & Chain Log

---

This notebook adds two features to your session:

1. **FFmpeg Video Effects** — Choose effects per scene (film grain, VHS, sepia, etc.) or let the machine randomize
2. **Chain Logger** — View a complete log of every AI decision the pipeline made

> **🤔** What aesthetic assumptions are baked into effects like "film grain" or "VHS artifacts"? Who decided what "cinematic" looks like?

---

In [ ]:
# SETUP
import json, random
from pathlib import Path
from IPython.display import display, Markdown, HTML
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
print('✅ Setup')

In [ ]:
# SELECT SESSION
sessions = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d/'session_info.json').exists()])
if not sessions: print('❌ No sessions!')
else:
    print('📂 Sessions:')
    for i,s in enumerate(sessions): print(f'   {i}: {s}')
SESSION_NAME = sessions[-1] if sessions else ''
SESSION_DIR = BASE_DIR / SESSION_NAME
print(f'✅ Session: {SESSION_NAME}')

---
## 🎬 Part 1: FFmpeg Video Effects

Choose an effect for each scene, or let the machine pick randomly.
Effects are applied during Assembly (06_Assemble).

In [ ]:
# ============================================================
# AVAILABLE EFFECTS (from the original ROTBOTS pipeline)
# ============================================================

EFFECTS = {
    'none':               'No effect (clean video)',
    'film_grain':         'Subtle photographic grain overlay',
    'vhs_artifacts':      'VHS tracking bands and color bleeding',
    'celluloid_scratches':'Film grain with random scratch lines',
    'sepia_tone':         'Warm sepia color grading',
    'bw_transition':      'Black and white',
    'color_grade_warm':   'Warm orange/amber color grade',
    'color_grade_cool':   'Cool blue/teal color grade',
    'vignette':           'Dark vignette around edges',
    'flicker':            'Brightness flicker (old projector)',
    'desaturate':         'Muted/desaturated colors',
}

print('🎬 Available effects:')
for k,v in EFFECTS.items(): print(f'   {k:25s} {v}')

In [ ]:
# ============================================================
# EFFECT SETTINGS
# ============================================================

ENABLE_EFFECTS = True

# Mode: 'random' = machine picks per scene, or set a fixed effect for all scenes
EFFECT_MODE = 'random'  # Options: 'random', or any effect name like 'film_grain', 'vhs_artifacts', etc.

# Per-scene overrides (optional): set specific effects for specific scenes
# Example: {2: 'sepia_tone', 3: 'vhs_artifacts', 4: 'film_grain'}
# Scene numbers match your prompts.json scenes. Leave empty to use EFFECT_MODE for all.
PER_SCENE_EFFECTS = {}

# Effect intensity (0.0 = invisible, 1.0 = full strength)
EFFECT_INTENSITY = 0.7

print(f'🎬 Effects: {"✅ enabled" if ENABLE_EFFECTS else "❌ disabled"}')
print(f'   Mode: {EFFECT_MODE}')
print(f'   Intensity: {EFFECT_INTENSITY}')
if PER_SCENE_EFFECTS: print(f'   Overrides: {PER_SCENE_EFFECTS}')

In [ ]:
# ============================================================
# ASSIGN EFFECTS TO SCENES
# ============================================================

prompts_file = SESSION_DIR / 'prompts.json'
if not prompts_file.exists():
    print('❌ No prompts.json! Run 02_Script_Writer first.')
else:
    with open(prompts_file) as f: prompts = json.load(f)
    
    effect_names = [k for k in EFFECTS.keys() if k != 'none']
    
    for p in prompts:
        sn = p['scene']
        if not ENABLE_EFFECTS:
            p['ffmpeg_effects'] = []
        elif sn in PER_SCENE_EFFECTS:
            eff = PER_SCENE_EFFECTS[sn]
            p['ffmpeg_effects'] = [eff] if eff != 'none' else []
        elif EFFECT_MODE == 'random':
            p['ffmpeg_effects'] = [random.choice(effect_names)]
        elif EFFECT_MODE != 'none':
            p['ffmpeg_effects'] = [EFFECT_MODE]
        else:
            p['ffmpeg_effects'] = []
        p['effect_intensity'] = EFFECT_INTENSITY
    
    # Save updated prompts
    with open(prompts_file, 'w') as f: json.dump(prompts, f, indent=2)
    
    print(f'✅ Effects assigned to {len(prompts)} scenes:')
    for p in prompts:
        eff = p.get('ffmpeg_effects', [])
        tag = eff[0] if eff else 'none'
        print(f'   Scene {p["scene"]}: {p["title"]:40s} → {tag} ({EFFECT_INTENSITY:.0%})')
    print(f'\n✅ prompts.json updated — 06_Assemble will apply these effects.')

---
## 📊 Part 2: Chain Logger — Every AI Decision

The chain log shows every LLM call the pipeline made: what was asked, what the system prompt said, and what the AI responded.

> **🤔** This is the "anatomy" of the content machine. Every entry is an automated decision.

In [ ]:
# ============================================================
# BUILD CHAIN LOG from session files
# ============================================================

chain = {
    'session': SESSION_NAME,
    'pipeline_config': {},
    'interactions': []
}

# Load session info
if (SESSION_DIR / 'session_info.json').exists():
    with open(SESSION_DIR / 'session_info.json') as f:
        chain['pipeline_config'] = json.load(f)

# Reconstruct interactions from saved files
if (SESSION_DIR / 'summaries.json').exists():
    with open(SESSION_DIR / 'summaries.json') as f: d = json.load(f)
    for s in d.get('sources', []):
        chain['interactions'].append({
            'stage': 'summarization',
            'purpose': f'Summarize source: {s["source"][:50]}',
            'input_preview': s['source'],
            'output_preview': s['summary'][:200] + '...',
            'output_length': len(s['summary'])
        })

if (SESSION_DIR / 'essay_script.json').exists():
    with open(SESSION_DIR / 'essay_script.json') as f: essay = json.load(f)
    chain['interactions'].append({
        'stage': 'outline',
        'purpose': 'Generate essay outline',
        'input_preview': f'Topic: {chain["pipeline_config"].get("topic","")}',
        'output_preview': f'Title: {essay.get("title","")}\nThesis: {essay.get("thesis","")}\nChapters: {len(essay.get("chapters", []))}',
        'output_length': len(json.dumps(essay))
    })
    for ch in essay.get('chapters', []):
        for sec in ch.get('sections', []):
            narr = sec.get('narration', '')
            chain['interactions'].append({
                'stage': 'chapter_writing',
                'purpose': f'Write Ch{ch["chapter"]}.{sec.get("section","?")}',
                'input_preview': ch['title'],
                'output_preview': narr[:150],
                'output_length': len(narr),
                'word_count': len(narr.split())
            })

if (SESSION_DIR / 'prompts.json').exists():
    with open(SESSION_DIR / 'prompts.json') as f: prompts_data = json.load(f)
    for p in prompts_data:
        chain['interactions'].append({
            'stage': 't2v_prompts',
            'purpose': f'Scene {p["scene"]}: {p["title"]}',
            'input_preview': f'Style: {p["style"]}',
            'output_preview': p['t2v_prompt'][:150],
            'output_length': len(p['t2v_prompt']),
            'effect': p.get('ffmpeg_effects', [])
        })

# Save chain log
with open(SESSION_DIR / 'chain_log.json', 'w') as f: json.dump(chain, f, indent=2)

print(f'📊 Chain log: {len(chain["interactions"])} AI interactions recorded')
for stage in ['summarization','outline','chapter_writing','t2v_prompts']:
    count = sum(1 for i in chain['interactions'] if i['stage'] == stage)
    if count: print(f'   {stage}: {count} calls')

In [ ]:
# ============================================================
# GENERATE HTML CHAIN REPORT
# ============================================================

html = '''<!DOCTYPE html><html><head><meta charset="UTF-8"><style>
body{font-family:system-ui;background:#1a1a2e;color:#eaeaea;padding:2rem;max-width:900px;margin:0 auto}
h1{background:linear-gradient(135deg,#e94560,#ff6b6b);-webkit-background-clip:text;-webkit-text-fill-color:transparent;font-size:2rem}
.stats{display:flex;gap:2rem;margin:1rem 0;padding:1rem;background:#16213e;border-radius:8px}
.stat-val{font-size:1.8rem;font-weight:bold;color:#e94560}.stat-lbl{color:#a0a0a0;font-size:.85rem}
.card{background:#16213e;border-radius:8px;padding:1rem;margin:1rem 0;border-left:4px solid #e94560}
.stage{background:#e94560;color:white;padding:2px 8px;border-radius:12px;font-size:.8rem;font-weight:600}
.purpose{color:#a0a0a0;margin-left:8px}
.box{background:#0d1b2a;padding:.8rem;border-radius:6px;font-family:monospace;font-size:.85rem;white-space:pre-wrap;margin:.5rem 0;max-height:200px;overflow-y:auto}
.in{border-left:3px solid #3498db}.out{border-left:3px solid #4ecca3}
details summary{cursor:pointer;font-weight:600;color:#4ecca3;margin:.3rem 0}
</style></head><body>
'''

html += f'<h1>🤖 ROTBOTS Chain Report</h1>'
html += f'<p style="color:#a0a0a0">Session: {SESSION_NAME}</p>'

# Stats
stages = set(i['stage'] for i in chain['interactions'])
total_chars = sum(i.get('output_length',0) for i in chain['interactions'])
html += f'<div class="stats">'
html += f'<div><div class="stat-val">{len(chain["interactions"])}</div><div class="stat-lbl">AI Queries</div></div>'
html += f'<div><div class="stat-val">{len(stages)}</div><div class="stat-lbl">Pipeline Stages</div></div>'
html += f'<div><div class="stat-val">{total_chars:,}</div><div class="stat-lbl">Chars Generated</div></div>'
html += '</div>'

# Interactions
html += '<h2 style="color:#e94560;margin-top:2rem">📊 Data Flow Timeline</h2>'
for idx, inter in enumerate(chain['interactions'], 1):
    html += f'<div class="card">'
    html += f'<span class="stage">{inter["stage"]}</span>'
    html += f'<span class="purpose">{inter["purpose"]}</span>'
    html += f'<details><summary>Input</summary><div class="box in">{inter.get("input_preview","")}</div></details>'
    html += f'<details open><summary>Output ({inter.get("output_length",0)} chars)</summary><div class="box out">{inter.get("output_preview","")}</div></details>'
    eff = inter.get('effect', [])
    if eff: html += f'<div style="color:#f39c12;margin-top:.3rem">🎬 Effect: {eff[0]}</div>'
    html += '</div>'

html += '</body></html>'

report_path = SESSION_DIR / 'chain_report.html'
with open(report_path, 'w', encoding='utf-8') as f: f.write(html)
print(f'✅ HTML report saved: {report_path}')
print(f'   Open from Google Drive to view in browser.')

In [ ]:
# PREVIEW CHAIN LOG (inline)
display(Markdown('# 📊 Chain Log Preview'))
display(Markdown(f'**Session:** {SESSION_NAME} | **AI Queries:** {len(chain["interactions"])} | **Stages:** {len(stages)}\n\n---'))
for i in chain['interactions']:
    eff = i.get('effect', [])
    eff_tag = f' | 🎬 {eff[0]}' if eff else ''
    display(Markdown(f'**[{i["stage"]}]** {i["purpose"]}{eff_tag}\n\n> {i.get("output_preview","")[:200]}\n\n---'))

---
## ⏭️ Next

Effects are saved in `prompts.json`. The Assembly notebook (06) will automatically detect and apply them.

The chain report is saved as `chain_report.html` in your session folder — open it in a browser for the full interactive view.

---
*ROTBOTS Workshop — LI-MA 2026*